<a href="https://colab.research.google.com/github/RisalKalwar/Starter-Notebooks/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RisalKalwar/Starter-Notebooks/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


## 1. My lane as an ML task

**Task type: Scoring / Ranking.**

This is not a simple classification problem (declining vs. not) on its own — the actual output
needed is a *ranked queue* of pages ordered by refresh priority, so a content team can work down
the list starting from the highest-value candidates. Under the hood this is built from a scoring
model (predicted probability of being a high-value refresh target), which is then used to rank.
It's the same shape as the starter pipeline: `03_train_model.py` outputs a score per page,
`04_evaluate_and_export.py` turns that into a ranked queue.
---



---



## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or proxy

There's no direct "should we refresh this" label in the data — no one has manually tagged pages
as refresh-worthy. So I need a **proxy label**.

**Proxy target:** a page is a positive example if it is currently declining (`trend_direction == "down"`)
AND still has meaningful visibility to recover (`impressions_90d` above a minimum threshold, so
we're not wasting effort on pages nobody sees anyway).

This proxy is imperfect — a page can decline for reasons a rewrite won't fix (e.g. the whole topic
lost demand). That's a known limitation, not a hidden one.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [11]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

def find_repo_root():
    cwd = os.getcwd()
    while True:
        if os.path.isdir(os.path.join(cwd, "data", "raw")):
            return cwd
        parent = os.path.dirname(cwd)
        if parent == cwd:
            return None
        cwd = parent

root = find_repo_root()
if root is None:
    if IN_COLAB:
        os.chdir("/content")
        if not os.path.isdir(REPO_DIR):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        os.chdir(REPO_DIR)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    os.chdir(root)

print("Working dir:", os.getcwd())

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Unit of analysis: one row = one page, observed over a 90-day window
print(f"{df.shape[0]} rows, {df.shape[1]} columns — one row per page")
df.head(3)



Working dir: /content/flyrank-ml-internship-starter
30000 rows, 44 columns — one row per page


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [12]:
# Build the proxy target column
min_impressions = 100  # threshold for "still has visibility worth recovering"

df["is_refresh_candidate"] = (
    (df["trend_direction"] == "down") & (df["impressions_90d"] >= min_impressions)
).astype(int)

print("Target distribution:")
print(df["is_refresh_candidate"].value_counts())
print(f"\nShare flagged as refresh candidates: {df['is_refresh_candidate'].mean():.1%}")

df[["trend_direction", "impressions_90d", "is_refresh_candidate"]].head(10)

Target distribution:
is_refresh_candidate
0    16848
1    13152
Name: count, dtype: int64

Share flagged as refresh candidates: 43.8%


,trend_direction,impressions_90d,is_refresh_candidate
0,down,3803,1
1,down,15320,1
2,down,12581,1
3,stable,11751,0
4,down,19140,1
5,down,3970,1
6,down,20,0
7,stable,1724,0
8,down,32574,1
9,down,1240,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML beats a fixed rule here

A fixed rule (e.g. "flag anything older than 12 months with declining traffic") only reaches about
24% precision on this data — three out of every four flagged pages turn out not to be worth
prioritizing. That's because refresh-worthiness depends on *combinations* of signals — position,
CTR relative to position tier, engagement, content age, traffic trend — that interact in ways a
single hand-written threshold can't capture. A learned model picking up on those interactions
reached roughly 3x the precision of the fixed rule in the starter pipeline. The gain isn't
theoretical — it's the difference between a content team wasting most of its refresh hours or not.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.